In [2]:
import gym
from gym import spaces
import numpy as np

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [1]:
from scipy.integrate import solve_ivp

In [20]:
class CoastingCarEnv(gym.Env):
    def __init__(self):
        super(CoastingCarEnv, self).__init__()

        # Define action and observation spaces
        self.action_space = spaces.Discrete(2)  # For simplicity, 0 for no action, 1 for braking
        self.observation_space = spaces.Box(low=np.array([0, 0, 0]), high=np.array([100, 100, 100]), dtype=np.float32)

        # Initialize state variables
        self.x = 0  # position of the car
        self.v = 10  # initial velocity of the car
        self.a = 0  # acceleration of the car
        self.m = 1000  # stop sign position
        self.t = 0  # clock variable

        # Constants
        self.b = 1  # braking capabilities
        self.ep = 0.1  # reaction time
        

    def step(self, action):

        if action == 1:  # Brake
            self.a = -self.b
        else:
            self.a = 0
        
        def model(t, y):
            x, v, t = y
            dydt = [v, self.a, 1]
            return dydt

        def zero_velocity_event(t_ode, y):
            v_state = y[1] # v is the second element (index 1) of the state vector y
            return v_state

        # We want the solver to stop when the event is found (terminal=True)
        # and we want it to detect the event regardless of the direction (direction=0)
        zero_velocity_event.terminal = True
        zero_velocity_event.direction = 0
        
        # 2. Solve the ODEs with the event
        # The integration will stop at t=self.t+1 OR when v=0, whichever comes first.
        sol = solve_ivp(
            model, 
            (self.t, self.t + 1), 
            [self.x, self.v, self.t], 
            method='RK45', 
            events=zero_velocity_event
        )
    
        # Update the state variables based on the integration result
        self.x = sol.y[0, -1]
        self.v = sol.y[1, -1]
        self.t = sol.y[2, -1] if self.t < sol.y[2, -1] else self.t + 1

        # Restriction
        if self.v < 0:
            self.v = 0

        # if self.t > self.ep:
        #     self.t = self.ep

        # Check if the car has reached the stop sign
        done = self.x >= self.m
        # print('done: ', done)

        #------------------------------------------
        if self.v**2 <= 2*self.b*(self.m-self.x):
          reward = 1
        else:
          # print(self.v**2, 2*self.b*(self.m-self.x))
          reward = -10
        #--------------------------------------------

        return np.array([self.x, self.v, self.t]), reward, done, {}

    def reset(self):
        # Reset state variables
        self.x = 0
        self.v = 10
        self.a = 0
        self.t = 0

        return np.array([self.x, self.v, self.t])

    def render(self, mode='human'):
        print(f"Position: {self.x}, Velocity: {self.v}, Time: {self.t}")

    def close(self):
        pass



In [21]:
# Test the environment
env = CoastingCarEnv()
obs = env.reset()
done = False
for i in range(100):
    action = 1  # Brake
    # print("Before: ")
    # env.render()
    obs, reward, done, _ = env.step(action)
    # print("After")
    env.render()
    print(reward)
    

Position: 9.500000000000002, Velocity: 9.0, Time: 0.9999999999999998
1
Position: 18.000000000000007, Velocity: 8.0, Time: 1.9999999999999996
1
Position: 25.500000000000007, Velocity: 7.000000000000001, Time: 2.999999999999999
1
Position: 32.00000000000001, Velocity: 6.000000000000002, Time: 3.999999999999999
1
Position: 37.500000000000014, Velocity: 5.000000000000002, Time: 4.999999999999999
1
Position: 42.000000000000014, Velocity: 4.000000000000002, Time: 5.999999999999999
1
Position: 45.50000000000002, Velocity: 3.0000000000000018, Time: 6.999999999999999
1
Position: 48.00000000000003, Velocity: 2.0000000000000018, Time: 7.999999999999999
1
Position: 49.500000000000036, Velocity: 1.000000000000001, Time: 9.0
1
Position: 50.000000000000036, Velocity: 1.3322676295501878e-15, Time: 10.0
1
Position: 50.000000000000036, Velocity: 1.3322676295501878e-15, Time: 11.0
1
Position: 50.000000000000036, Velocity: 1.3322676295501878e-15, Time: 12.0
1
Position: 50.000000000000036, Velocity: 1.3322

d:\Reinforcement learning\kyx_parser\.venv\Lib\site-packages\gym\spaces\box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")
